<a href="https://colab.research.google.com/github/Eshikasanjana/Hybrid-Summarization-of-Legal-Documents/blob/main/legal_summarizer_hybrid_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q -U pyarrow datasets google-genai transformers rouge sentence-transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.


In [2]:
from google.colab import userdata
import os

In [3]:
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [4]:
from huggingface_hub import notebook_login
notebook_login()
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [5]:
import os
import nltk
import torch
import pandas as pd
from rouge import Rouge
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from nltk.cluster import KMeansClusterer
from scipy.spatial import distance_matrix
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [6]:
import os
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [7]:
from google import genai
client = genai.Client()

In [8]:
nltk.download('punkt')
nltk.download('punkt_tab')

# Device setup
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Using device: cpu


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [9]:
dataset = load_dataset("FiscalNote/billsum")
test_cases = dataset["ca_test"]
subset = test_cases.select(range(1))
texts = subset["text"]
gold_summaries = subset["summary"]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

In [10]:
embedder = SentenceTransformer('distiluse-base-multilingual-cased')
pegasus_model_name = "nsi319/legal-pegasus"
tokenizer = AutoTokenizer.from_pretrained(pegasus_model_name)
pegasus_model = AutoModelForSeq2SeqLM.from_pretrained(pegasus_model_name).to(device)

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/683 [00:00<?, ?it/s]

In [11]:
def kmeans_extractive_summary(text, num_clusters=10, iterations=25):
    sentences = nltk.sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    if len(sentences) == 0:
        return ""
    k = min(num_clusters, len(sentences))
    embeddings = embedder.encode(sentences)
    kclusterer = KMeansClusterer(
        k,
        distance=nltk.cluster.util.cosine_distance,
        repeats=iterations,
        avoid_empty_clusters=True)
    assigned = kclusterer.cluster(embeddings, assign_clusters=True)
    centroids = kclusterer.means()
    tempdata = pd.DataFrame({
        "sentence": sentences,
        "embeddings": list(embeddings),
        "cluster": assigned})
    tempdata["centroid"] = tempdata["cluster"].apply(lambda x: centroids[x])
    tempdata["dist"] = tempdata.apply(lambda r: distance_matrix([r["embeddings"]], [r["centroid"]])[0][0], axis=1)
    selected = (tempdata.sort_values("dist", ascending=True)
                .groupby("cluster")
                .head(1)
                .sort_index()["sentence"].tolist())
    return " ".join(selected)


In [12]:
def run_pegasus_summary(model, tokenizer, text, max_length=512, num_beams=9):
    inputs = tokenizer.encode(text, truncation=True, padding="longest", return_tensors="pt").to(device)
    summary_ids = model.generate(inputs, max_length=max_length, num_beams=num_beams, early_stopping=True)
    decoded = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
    return decoded[0]

In [13]:
def summarize_text(text, num_clusters=10, iterations=25, pegasus_max_length=512, pegasus_num_beams=9):
    sentences = [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]
    if not sentences:
        return {"extractive_summary": "", "pegasus_full_summary": "", "hybrid_summary": ""}
    safe_k = min(num_clusters, len(sentences))
    extractive = kmeans_extractive_summary(text, num_clusters=safe_k, iterations=iterations)
    pegasus_full = run_pegasus_summary(pegasus_model, tokenizer, text, max_length=pegasus_max_length, num_beams=pegasus_num_beams)
    if extractive.strip():
        hybrid = run_pegasus_summary(pegasus_model, tokenizer, extractive, max_length=pegasus_max_length, num_beams=pegasus_num_beams)
    else:
        hybrid = ""
    return {
        "extractive_summary": extractive,
        "pegasus_full_summary": pegasus_full,
        "hybrid_summary": hybrid,
    }


In [16]:
def create_evaluation_prompt(original, summary):
    return f"""
You are a legal expert. Evaluate the given summary for:
1. Factual accuracy
2. Coverage of important points
3. Clarity and readability

Original legal text:
{original}

Summary:
{summary}

Please provide scores from 1 to 5 on each aspect and a brief explanation.
"""

# Call Gemini API to evaluate summary
def evaluate_summary_with_gemini(original_text, summary_text):
    prompt = create_evaluation_prompt(original_text, summary_text)
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text

# Main execution: input, summarize, evaluate
user_text = input("Paste the bill text to summarize, then press Enter:\n")

summaries = summarize_text(user_text)

print("\n Extractive Summary \n")
print(summaries["extractive_summary"])

print("\n Pegasus Full Summary \n")
print(summaries["pegasus_full_summary"])

print("\n Hybrid (Extractive → Pegasus) Summary \n")
print(summaries["hybrid_summary"])

evaluation = evaluate_summary_with_gemini(user_text, summaries["hybrid_summary"])
print("\n Gemini Evaluation of Hybrid Summary \n")
print(evaluation)

Paste the bill text to summarize, then press Enter:
A rent or lease agreement is required to be printed on stamp paper of correct value, as per laws in different States. If the lease agreement is up to 11 months, it does not mandatorily need registration. However, a rent agreement more than 11 months is required to be registered.

 Extractive Summary 

A rent or lease agreement is required to be printed on stamp paper of correct value, as per laws in different States. If the lease agreement is up to 11 months, it does not mandatorily need registration. However, a rent agreement more than 11 months is required to be registered.

 Pegasus Full Summary 

A rent or lease agreement is required to be printed on stamp paper of correct value, as per laws in different States.<n>If the lease agreement is up to 11 months, it does not need registration. However, a rent agreement more than 11 months is required to be registered.

 Hybrid (Extractive → Pegasus) Summary 

A rent or lease agreement is